In [ ]:
import re
CATALOG = dbutils.widgets.get("CATALOG")
COMPLAINT_LAKEBASE_INSTANCE_NAME = re.sub(r'[^a-z0-9-]', '-', f"{CATALOG}complaintmanager".lower())

In [ ]:
%pip install databricks-sdk --upgrade

In [ ]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.database import DatabaseInstance
from databricks.sdk.errors import NotFound, ResourceDoesNotExist
import time

import sys
sys.path.append('../utils')
from uc_state import add

w = WorkspaceClient()

instance_name = COMPLAINT_LAKEBASE_INSTANCE_NAME

try:
    existing = w.database.get_database_instance(instance_name)
    print(f"♻️ Found existing database instance: {existing.name} (state: {existing.state})")
    instance = existing
except (NotFound, ResourceDoesNotExist):
    print(f"Creating new database instance: {instance_name}")
    instance = w.database.create_database_instance(
        DatabaseInstance(
            name=instance_name,
            capacity="CU_1"
        )
    )
    print(f"Created database instance: {instance.name}")
    add(CATALOG, "databaseinstances", {"name": instance.name, "type": "complaint"})

print("Waiting for database instance to be available...")
for _ in range(60):
    state = str(w.database.get_database_instance(instance_name).state)
    if state == "DatabaseInstanceState.AVAILABLE":
        print(f"Instance ready: {state}")
        break
    time.sleep(10)
else:
    raise TimeoutError(f"Database instance {instance_name} did not become available within 10 minutes")

In [ ]:
# create a database `caspers_complaints` in the lakebase instance
# register it to UC as concat of CATALOG-INSTANCE_NAME (params)

from databricks.sdk.service.database import DatabaseCatalog

catalog_name = COMPLAINT_LAKEBASE_INSTANCE_NAME

try:
    existing_cat = w.catalogs.get(catalog_name)
    print(f"♻️ Found existing catalog: {existing_cat.name}")
except Exception:
    catalog = w.database.create_database_catalog(
        DatabaseCatalog(
            name=catalog_name,
            database_instance_name=COMPLAINT_LAKEBASE_INSTANCE_NAME,
            database_name="caspers_complaints",
            create_database_if_not_exists=True
        )
    )
    print(f"Created new database and catalog: {catalog.name}")
    add(CATALOG, "databasecatalogs", catalog)

In [ ]:
from databricks.sdk.service.database import SyncedDatabaseTable, SyncedTableSpec, NewPipelineSpec, SyncedTableSchedulingPolicy

synced_table_name = f"{CATALOG}.complaints.pg_complaint_responses"

try:
    existing_table = w.tables.get(synced_table_name)
    print(f"♻️ Found existing synced table: {existing_table.full_name}")
except Exception:
    synced_table = w.database.create_synced_database_table(
        SyncedDatabaseTable(
            name=synced_table_name,
            database_instance_name=instance.name,
            logical_database_name="caspers_complaints",
            spec=SyncedTableSpec(
                source_table_full_name=f"{CATALOG}.complaints.complaint_responses",
                primary_key_columns=["complaint_id"],
                scheduling_policy=SyncedTableSchedulingPolicy.CONTINUOUS,
                create_database_objects_if_missing=True,
                # P1-19: co-locate the synced-table pipeline's backing storage with
                # the source table's schema (`{CATALOG}.complaints`).  The previous
                # literal "storage_catalog"/"storage_schema" placeholders would resolve
                # to non-existent UC names on any fresh deploy.
                new_pipeline_spec=NewPipelineSpec(
                    storage_catalog=CATALOG,
                    storage_schema="complaints",
                )
            ),
        )
    )
    print(f"Created synced table: {synced_table.name}")
    add(CATALOG, "pipelines", synced_table.data_synchronization_status)